# Código de cálculo

Cálculos realizados na Iniciação, além de um tratamento de dados sobre as eficiências possíveis, além dos casos que são possíveis físicamente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import fminbound



# Cálculos necessários:
- Energia interna;
- Calor Total
- Variação da energia
- $C_{p}$ médio
- Ponto do regenerador e otimizar este ponto.
- Cálculo das eficiências
- Filtragem das eficiências de carnot, e temperatura.

In [ ]:
def clc_eff(df):
    #Energia interna:
    df['U1B'] = df['H1B'] - df['P1B']/ df['RHO1B']
    df['U2B'] = df['H2B'] - df['P2B']/ df['RHO2B']
    df['U3B'] = df['H3B'] - df['P3B']/ df['RHO3B']
    df['U4B'] = df['H4B'] - df['P4B']/ df['RHO4B']

    #Calor total:
    df['QCOMP_total'] = df['T1B']*(df['S1B']- df['S4B']) * df['MCICLO']
    df['QTURB_total'] = df['T3B']*(df['S3B']- df['S2B']) * df['MCICLO']
    df['QAQUEC_total'] = df['MCICLO'] * (df['U2B'] - df['U1B'])
    df['QRESF_total'] = df['MCICLO'] * (df['U4B'] - df['U3B'])

    #Variação nas energias:
    df['deltaU_COMP'] = (df['U1B'] - df['U4B']) * df['MCICLO']
    df['deltaU_TURB'] = (df['U3B'] - df['U2B']) * df['MCICLO']
    df['WCOMP'] = df['QCOMP_total'] - df['deltaU_COMP']
    df['WTURB'] = df['QTURB_total'] - df['deltaU_TURB']
    df['WNET'] = abs(df['WTURB'] - df['WCOMP'])

    #Cálculo do Cp médio
    df['Cp3B'] = (df['H4B'] - df['H3B']) / (df['T4B'] - df['T3B'])
    df['Cp1B'] = (df['H2B'] - df['H1B']) / (df['T2B'] - df['T1B'])


    #arrays
    Cp3B_arr = df['Cp3B'].values
    Cp1B_arr = df['Cp1B'].values
    T3B_arr = df['T3B'].values
    T1B_arr = df['T1B'].values
    Mciclo_arr = df['MCICLO'].values
    Qturb_arr = df['QTURB_total'].values
    Qaquec_arr = df['QAQUEC_total'].values
    Wnet_arr = df['WNET'].values
    def f_pinch(DeltaT_pinch):
        T1b_int = (Cp3B_arr *(T3B_arr - DeltaT_pinch) + Cp1B_arr*T1B_arr) / (Cp1B_arr + Cp3B_arr)
        Qregen = Mciclo_arr *Cp1B_arr *(T1b_int -T1B_arr)
        Qregen = np.clip(Qregen, 0, None)

        Qreal =Qturb_arr +Qaquec_arr -Qregen
        Ef_regen = (Wnet_arr /Qreal) *100

        valid_mask = (Ef_regen >= 0) & (Ef_regen <= 100)
        Ef_regen_valid =np.where(valid_mask,Ef_regen, np.nan)

        return -np.nanmean(Ef_regen_valid)

    DeltaT_otimo = fminbound(f_pinch, 0.1, 30)
    Ef_otimo = -f_pinch(DeltaT_otimo)

    print(f'DeltaT ótimo: {DeltaT_otimo} K, Eficiência de {Ef_otimo}%')

    #aplicar e cálcular eficiências
    df['T1B_int'] = (
        df['Cp3B'] * (df['T3B'] - DeltaT_otimo) + df['Cp1B'] * df['T1B']
    ) / (df['Cp1B'] + df['Cp3B'])
    df['T3B_int'] = df['T1B_int'] + DeltaT_otimo
    df['Qregen'] = (df['MCICLO'] * df['Cp1B'] * (df['T1B_int'] - df['T1B'])).clip(lower=0)
    df['Qreal'] = df['QAQUEC_total'] - df['Qregen']

    df['Efcarnot'] = (1 - df['T1B'] / df['T3B']) * 100
    df['Ef_regen'] = (df['WNET'] / (df['QTURB_total'] + df['Qreal'])) * 100
    df['Ef_sregen'] = (df['WNET'] / (df['QTURB_total'] + df['QAQUEC_total'])) * 100

    mask = (
        (df['Efcarnot'] >= 0) & (df['Efcarnot'] <= 100) &
        (df['Ef_regen'] >= 0) & (df['Ef_regen'] <= 100) &
        (df['Ef_sregen'] >= 0) & (df['Ef_sregen'] <= 100) &
        (df['Ef_regen'] <= df['Efcarnot']) &
        (df['Ef_sregen'] <= df['Efcarnot'])
    )

    #DEBUG: Quantas linhas passaram no filtro:
    print(f''' \n=== DEBUG ===
    Total de linhas originais: {len(df)}
    Linhas que passaram no filtro: {mask.sum()}
    \nAnálise de cada critério:
    Efcarnot válida (0-100): {((df['Efcarnot'] >= 0) & (df['Efcarnot'] <= 100)).sum()}
    Ef_regen válida (0-100): {((df['Ef_regen'] >= 0) & (df['Ef_regen'] <= 100)).sum()}
    Efcarnot válida (0-100): {((df['Efcarnot'] >= 0) & (df['Efcarnot'] <= 100)).sum()}
    Ef_regen <= Efcarnot: {(df['Ef_regen'] <= df['Efcarnot']).sum()}
    Ef_sregen <= Efcarnot: {(df['Ef_sregen'] <= df['Efcarnot']).sum()}
    \nValores das Eficiências:
    Efcarnot: min={df['Efcarnot'].min():.2f}, max={df['Efcarnot'].max():.2f}
    Ef_regen: min={df['Ef_regen'].min():.2f}, max={df['Ef_regen'].max():.2f}
    Ef_sregen: min={df['Ef_sregen'].min():.2f}, max={df['Ef_sregen'].max():.2f}
    =============\n
''')

    df_filtrado = df[mask].copy()

    if len(df_filtrado) == 0:
        print("Aviso nenhuma linha passou no filtro! Exportando dados sem filtro.")
        df_filtrado = df.copy()

    return df_filtrado, DeltaT_otimo, Ef_otimo

# Plotagem dos gráficos do sistema
- Plotar mapa de temperatura;
- Plotar mapa de pressão;

Ambos os mapas com base no aquecedor e resfriador, como no expansor e no compressor.

In [ ]:
def plot_eff_maps(df, col_x, col_y, eff_col, xlabel, ylabel, title, output_name, save_pdf=False):
    df_pivot = df.pivot_table(index=col_y, columns=col_x, values=eff_col, aggfunc='mean')
    x_vals = df_pivot.columns.values
    y_vals = df_pivot.index.values
    Z = df_pivot.values
    Z = np.where(Z <= 0, np.nan, Z)

    fig, ax = plt.subplots(figsize=(10, 6))
    cp = ax.contourf(x_vals, y_vals, Z, levels=30, cmap='viridis')
    fig.colorbar(cp, label='Eficiência (%)')

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    plt.show()

    if save_pdf:
        caminho_salvamento = os.path.join(os.getcwd(), output_name)
        fig.savefig(caminho_salvamento, format='pdf', bbox_inches='tight')
        print(f'Gráfico salvo em: {caminho_salvamento}')

    plt.close(fig)

In [ ]:
import os
df = pd.read_excel('dados.xlsx')

df_filtrado, deltaT, ef_opt =clc_eff(df)


caminho_projeto =os.path.abspath(os.getcwd())
arquivo_saida = os.path.join(caminho_projeto,'Resultado_eficiencia.xlsx')
with pd.ExcelWriter(arquivo_saida) as writer:
    df_filtrado.to_excel(writer, sheet_name='Geral', index=False)
    for Tfix in df_filtrado['T4B'].unique():
        df_filtrado[df_filtrado['T4B'] == Tfix].to_excel(
            writer, sheet_name=f'Tresf_{int(Tfix)}K', index=False)
print(f'Arquivo {arquivo_saida} criado com todas as eficiências.')

In [ ]:
plot_eff_maps(
    df=df_filtrado,
    col_x='Pturb',
    col_y='Pcomp',
    eff_col='Ef_regen',
    xlabel='Pressão de Turbina (bar)',
    ylabel='Pressão de Compressão (bar)',
    title='Mapa de eficiência, pressão, regenerativo',
    output_name='mapa_pressao_regenerativo.pdf',
    save_pdf=True
)
plot_eff_maps(
    df=df_filtrado,
    col_x='Pturb',
    col_y='Pcomp',
    eff_col='Ef_sregen',
    xlabel='Pressão de Turbina (bar)',
    ylabel='Pressão de Compressão (bar)',
    title='Mapa de eficiência, pressão, sem regeneração',
    output_name='mapa_pressao_sem_regeneracao.pdf',
    save_pdf=True
)
plot_eff_maps(
    df=df_filtrado,
    col_x='T3B',
    col_y='T1B',
    eff_col='Ef_regen',
    xlabel='Temperatura Quente (K)',
    ylabel='Temperatura Frio (K)',
    title='Mapa de eficiência, temperatura, regenerativo',
    output_name='mapa_temperatura_regenerativo.pdf',
    save_pdf=True
)
plot_eff_maps(
    df=df_filtrado,
    col_x='T3B',
    col_y='T1B',
    eff_col='Ef_sregen',
    xlabel='Temperatura Quente (K)',
    ylabel='Temperatura Frio (K)',
    title='Mapa de eficiência, temperatura, sem regeneração',
    output_name='mapa_temperatura_sem_regeneracao.pdf',
    save_pdf=True
)

**Teste para ver onde os arquivos estão sendo exportados, após serem cálculados!!**

In [ ]:
import os
# Isso vai te mostrar exatamente a pasta onde o Python criou o arquivo
print(f"Pasta onde o Python está trabalhando: {os.getcwd()}")

# Isso vai procurar os PDFs especificamente nessa pasta
arquivos_na_pasta = os.listdir(os.getcwd())
pdfs_encontrados = [f for f in arquivos_na_pasta if f.endswith('.pdf')]
print(f"PDFs encontrados nesta pasta: {pdfs_encontrados}")